# Experiment

In [2]:
import pyterrier as pt
import pandas as pd
import os
from pathlib import Path

In [3]:
dataset_name = "irds:cord19/trec-covid/round5"
dataset = pt.get_dataset(dataset_name)

## Index Creation/Loading

In [4]:
index_dir_path = Path("index/trec_covid_index").resolve()
index_dir = str(index_dir_path)
index_properties_file = os.path.join(index_dir, "data.properties")

In [5]:
os.makedirs(index_dir, exist_ok=True)

In [6]:
def trec_covid_corpus_generator():
    for doc in dataset.get_corpus_iter():
        title = str(doc.get('title', ''))
        abstract = str(doc.get('abstract', ''))
        
        yield {
            'docno': doc['docno'],
            'text': title + " " + abstract
        }

In [7]:
if not pt.started():
    pt.init()

C:\Users\zosia\AppData\Local\Temp\ipykernel_46456\2550260977.py:1: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
C:\Users\zosia\AppData\Local\Temp\ipykernel_46456\2550260977.py:2: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [8]:
if os.path.exists(index_properties_file):
    print(f"Index found at {index_dir}. Loading it...")
    index_ref = pt.IndexRef.of(index_properties_file)
else:
    print(f"Index not found at {index_dir}. Building it now (this may take a few minutes)...")
    
    indexer = pt.IterDictIndexer(index_dir, blocks=True, meta={'docno': 50})
    index_ref = indexer.index(trec_covid_corpus_generator())
    print("Index built successfully.")

Index not found at C:\Users\zosia\Desktop\Studies\Information Retrieval\IR_Project\src\index\trec_covid_index. Building it now (this may take a few minutes)...


cord19/trec-covid/round5 documents:   0%|          | 0/192509 [00:00<?, ?it/s]

14:24:24.436 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (8is9x9sc) - further warnings are suppressed
14:25:26.020 [main] ERROR org.terrier.structures.indexing.Indexer -- Could not finish MetaIndexBuilder: 
java.io.IOException: Key 8lqzfj2e is not unique: 37597,11755
For MetaIndex, to suppress, set metaindex.compressed.reverse.allow.duplicates=true
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.mergeTwo(FSOrderedMapFile.java:1374)
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.close(FSOrderedMapFile.java:1308)
	at org.terrier.structures.indexing.BaseMetaIndexBuilder.close(BaseMetaIndexBuilder.java:321)
	at org.terrier.structures.indexing.classical.BasicIndexer.indexDocuments(BasicIndexer.java:270)
	at org.terrier.structures.indexing.classical.BasicIndexer.createDirectIndex(BasicIndexer.java:388)
	at org.terrier.structures.indexing.Indexer.index(Indexer.java:377)
14:25:34.299 [main] WA

## Different Query Verbosity Levels

In [9]:
topics_data = []
for query in dataset.irds_ref().queries_iter():
    topics_data.append({
        'qid': query.query_id,
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })
df_all_topics = pd.DataFrame(topics_data)

In [10]:
# TITLE Queries
topics_title = df_all_topics[['qid', 'title']].copy()
topics_title = topics_title.rename(columns={'title': 'query'})

# DESCRIPTION Queries
topics_desc = df_all_topics[['qid', 'description']].copy()
topics_desc = topics_desc.rename(columns={'description': 'query'})

# NARRATIVE Queries
topics_narr = df_all_topics[['qid', 'narrative']].copy()
topics_narr = topics_narr.rename(columns={'narrative': 'query'})

## Models

In [11]:
bm25 = pt.terrier.Retriever(index_ref, wmodel="BM25")
ql_dir = pt.terrier.Retriever(index_ref, wmodel="DirichletLM")
ql_jm = pt.terrier.Retriever(index_ref, wmodel="Hiemstra_LM")
rm3_pipe = bm25 >> pt.rewrite.RM3(index_ref) >> bm25

In [12]:
models = [bm25, ql_dir, ql_jm, rm3_pipe]
model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3"]

## Experiment

In [13]:
eval_metrics = [
    "ndcg_cut_10",
    "P_5",
    "recip_rank",
    "recall_100",
    "map"
]
query_variants = {
    "Title": topics_title,
    "Description": topics_desc,
    "Narrative": topics_narr
}

In [14]:
for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=model_names,
        baseline=0
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.102569,0.440615,0.284,0.114277,0.264559,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.084593,0.318457,0.204,0.100734,0.167425,10.0,40.0,0.000789,12.0,...,0.017886,8.0,22.0,0.010926,16.0,31.0,0.007409,7.0,33.0,0.000194
2,QL (JM),0.067670,0.431482,0.228,0.086761,0.201066,5.0,45.0,0.000002,15.0,...,0.877984,11.0,22.0,0.127879,6.0,38.0,0.000020,14.0,29.0,0.030831
3,BM25 + RM3,0.105599,0.467502,0.272,0.113671,0.247447,28.0,22.0,0.345166,17.0,...,0.357729,12.0,12.0,0.606512,21.0,22.0,0.803250,20.0,21.0,0.353247




Results for DESCRIPTION Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.104567,0.424163,0.232,0.122201,0.210341,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.085222,0.370318,0.192,0.101649,0.163666,12.0,38.0,9.545769e-05,23.0,...,0.329193,12.0,18.0,0.150782,14.0,35.0,0.000161,14.0,30.0,0.027155
2,QL (JM),0.059483,0.345951,0.188,0.089350,0.168100,3.0,47.0,9.374414e-09,16.0,...,0.214295,11.0,19.0,0.207202,7.0,41.0,0.000098,15.0,28.0,0.136207
3,BM25 + RM3,0.113141,0.421349,0.280,0.124494,0.247513,30.0,20.0,5.899559e-02,15.0,...,0.913972,13.0,10.0,0.122430,19.0,21.0,0.592806,24.0,17.0,0.088994




Results for NARRATIVE Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.075474,0.398526,0.276,0.086866,0.224339,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.044626,0.220276,0.132,0.067869,0.099810,6.0,44.0,0.000002,12.0,...,0.002098,7.0,22.0,0.000830,8.0,37.0,0.000010,9.0,30.0,0.000069
2,QL (JM),0.058445,0.296964,0.188,0.074529,0.167937,9.0,41.0,0.000184,12.0,...,0.020902,4.0,18.0,0.002918,14.0,32.0,0.030330,12.0,24.0,0.012988
3,BM25 + RM3,0.083607,0.366773,0.216,0.085393,0.201771,21.0,29.0,0.227443,9.0,...,0.121168,4.0,14.0,0.020579,16.0,27.0,0.757486,14.0,23.0,0.142053


## Experiment with Dense Retrieval (DPR)

In [16]:
!pip install -q torch torchvision torchaudio transformers
!pip install -q faiss-cpu 
!pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_dr.git


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import pyterrier_dr as ptdr
import os
from pathlib import Path

dense_index_dir = str(Path("index/trec_covid_dense").resolve())

if os.path.exists(os.path.join(dense_index_dir, "pt_meta.json")):
    print("Found downloaded dense index! Loading...")
    dpr_model = ptdr.SBertBiEncoder('sentence-transformers/all-MiniLM-L6-v2')
    dense_index = ptdr.FlexIndex(dense_index_dir)
    dpr_pipe = dpr_model >> dense_index
    
    print("DPR Pipeline ready!")
else:
    print(f"Could not find the dense index at {dense_index_dir}. Please check where you extracted it!")

Found downloaded dense index! Loading...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DPR Pipeline ready!


In [18]:
all_models = [bm25, ql_dir, ql_jm, rm3_pipe, dpr_pipe]
all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "DPR (all-MiniLM)"]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0 
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.807094,0.672,0.106150,0.601847,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.684478,0.540,0.094298,0.444853,14.0,36.0,3.445424e-04,5.0,...,0.011665,8.0,23.0,7.844063e-04,16.0,33.0,5.811970e-03,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.657621,0.452,0.075799,0.389103,2.0,48.0,2.471805e-09,7.0,...,0.018710,5.0,30.0,2.983346e-06,5.0,44.0,4.825223e-10,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.777866,0.684,0.109137,0.580049,30.0,20.0,1.205768e-01,4.0,...,0.202878,13.0,12.0,6.952988e-01,29.0,18.0,2.391932e-01,21.0,25.0,2.834572e-01
4,DPR (all-MiniLM),0.075150,0.569682,0.336,0.050606,0.300213,3.0,47.0,2.424788e-09,7.0,...,0.000895,3.0,34.0,1.087754e-08,5.0,43.0,1.140979e-08,6.0,43.0,2.638420e-09




Results for DESCRIPTION Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.874500,0.716,0.112934,0.623256,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.757972,0.592,0.101559,0.505937,9.0,41.0,2.067988e-05,7.0,...,0.037531,7.0,23.0,0.001941,16.0,29.0,2.971897e-03,15.0,35.0,0.000169
2,QL (JM),0.125899,0.715361,0.528,0.078242,0.444361,1.0,49.0,1.163376e-11,5.0,...,0.006109,6.0,30.0,0.000172,5.0,44.0,2.482803e-09,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.842557,0.752,0.117835,0.655691,32.0,18.0,4.463815e-02,2.0,...,0.038771,13.0,10.0,0.192114,27.0,16.0,2.018099e-01,26.0,20.0,0.126493
4,DPR (all-MiniLM),0.119925,0.680775,0.528,0.075400,0.448695,6.0,44.0,2.219091e-09,5.0,...,0.004876,9.0,28.0,0.001194,8.0,41.0,1.290954e-06,12.0,38.0,0.000475




Results for NARRATIVE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629
4,DPR (all-MiniLM),0.111516,0.543542,0.424,0.069849,0.367265,14.0,36.0,2.399987e-03,9.0,...,0.023483,9.0,28.0,0.003070,13.0,36.0,0.004678,11.0,35.0,0.000372


# BioDPR

In [1]:
!pip install -q --upgrade torch torchvision torchaudio


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import pyterrier_dr as ptdr
import os
from pathlib import Path

biodpr_index_dir = str(Path("index/trec_covid_biodpr").resolve())

if os.path.exists(os.path.join(biodpr_index_dir, "pt_meta.json")):
    print("Found BioDPR index! Loading...")
    
    biodpr_model = ptdr.SBertBiEncoder("pritamdeka/S-PubMedBert-MS-MARCO")
    biodpr_index = ptdr.FlexIndex(biodpr_index_dir)
    biodpr_pipe = biodpr_model >> biodpr_index
    
    print("BioDPR Pipeline ready!")
else:
    print(f"Could not find the index at {biodpr_index_dir}. Please make sure you extracted the folder correctly!")

Found BioDPR index! Loading...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BioDPR Pipeline ready!


In [16]:
all_models = [bm25, ql_dir, ql_jm, rm3_pipe, biodpr_pipe]
all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "BioDPR"]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0 
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.102569,0.440615,0.284,0.114277,0.264559,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.084593,0.318457,0.204,0.100734,0.167425,10.0,40.0,0.000789,12.0,...,0.017886,8.0,22.0,0.010926,16.0,31.0,7.408635e-03,7.0,33.0,0.000194
2,QL (JM),0.067670,0.431482,0.228,0.086761,0.201066,5.0,45.0,0.000002,15.0,...,0.877984,11.0,22.0,0.127879,6.0,38.0,1.969378e-05,14.0,29.0,0.030831
3,BM25 + RM3,0.105599,0.467502,0.272,0.113671,0.247447,28.0,22.0,0.345166,17.0,...,0.357729,12.0,12.0,0.606512,21.0,22.0,8.032496e-01,20.0,21.0,0.353247
4,BioDPR,0.049212,0.285959,0.200,0.071744,0.159773,9.0,41.0,0.000004,12.0,...,0.004565,8.0,23.0,0.031525,6.0,42.0,1.652845e-07,10.0,31.0,0.001501




Results for DESCRIPTION Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.104567,0.424163,0.232,0.122201,0.210341,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.085222,0.370318,0.192,0.101649,0.163666,12.0,38.0,9.545769e-05,23.0,...,0.329193,12.0,18.0,0.150782,14.0,35.0,0.000161,14.0,30.0,0.027155
2,QL (JM),0.059483,0.345951,0.188,0.089350,0.168100,3.0,47.0,9.374414e-09,16.0,...,0.214295,11.0,19.0,0.207202,7.0,41.0,0.000098,15.0,28.0,0.136207
3,BM25 + RM3,0.113141,0.421349,0.280,0.124494,0.247513,30.0,20.0,5.899559e-02,15.0,...,0.913972,13.0,10.0,0.122430,19.0,21.0,0.592806,24.0,17.0,0.088994
4,BioDPR,0.084821,0.383279,0.256,0.107315,0.227550,19.0,31.0,1.984655e-02,25.0,...,0.522290,20.0,17.0,0.549807,18.0,28.0,0.077227,26.0,20.0,0.568853




Results for NARRATIVE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.075474,0.398526,0.276,0.086866,0.224339,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.044626,0.220276,0.132,0.067869,0.099810,6.0,44.0,0.000002,12.0,...,0.002098,7.0,22.0,0.000830,8.0,37.0,0.000010,9.0,30.0,0.000069
2,QL (JM),0.058445,0.296964,0.188,0.074529,0.167937,9.0,41.0,0.000184,12.0,...,0.020902,4.0,18.0,0.002918,14.0,32.0,0.030330,12.0,24.0,0.012988
3,BM25 + RM3,0.083607,0.366773,0.216,0.085393,0.201771,21.0,29.0,0.227443,9.0,...,0.121168,4.0,14.0,0.020579,16.0,27.0,0.757486,14.0,23.0,0.142053
4,BioDPR,0.073757,0.402035,0.264,0.091069,0.242682,21.0,29.0,0.827853,21.0,...,0.952202,16.0,15.0,0.745559,22.0,24.0,0.523314,26.0,17.0,0.533645


# Hybrid Approach

In [17]:
import pyterrier as pt

def normalize_scores(res):
    res = res.copy()
    
    grouped = res.groupby('qid')['score']
    min_score = grouped.transform('min')
    max_score = grouped.transform('max')
    
    res['score'] = (res['score'] - min_score) / (max_score - min_score).replace(0, 1e-5)
    
    return res

norm_transformer = pt.apply.generic(normalize_scores)

bm25_norm = bm25 >> norm_transformer
biodpr_norm = biodpr_pipe >> norm_transformer

hybrid_pipe = bm25_norm + biodpr_norm

all_models = [bm25, ql_dir, ql_jm, rm3_pipe, biodpr_pipe, hybrid_pipe]
all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "BioDPR", "Hybrid (BM25 + BioDPR)"]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


C:\Users\zosia\anaconda3\Lib\site-packages\pyterrier\_evaluation\_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #5: Hybrid (BM25 + BioDPR) (<pyterrier._ops.Sum object at 0x0000022818C1BCE0>)
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.102569,0.440615,0.284,0.114277,0.264559,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.084593,0.318457,0.204,0.100734,0.167425,10.0,40.0,0.000789,12.0,...,0.017886,8.0,22.0,0.010926,16.0,31.0,7.408635e-03,7.0,33.0,0.000194
2,QL (JM),0.067670,0.431482,0.228,0.086761,0.201066,5.0,45.0,0.000002,15.0,...,0.877984,11.0,22.0,0.127879,6.0,38.0,1.969378e-05,14.0,29.0,0.030831
3,BM25 + RM3,0.105599,0.467502,0.272,0.113671,0.247447,28.0,22.0,0.345166,17.0,...,0.357729,12.0,12.0,0.606512,21.0,22.0,8.032496e-01,20.0,21.0,0.353247
4,BioDPR,0.049212,0.285959,0.200,0.071744,0.159773,9.0,41.0,0.000004,12.0,...,0.004565,8.0,23.0,0.031525,6.0,42.0,1.652845e-07,10.0,31.0,0.001501
5,Hybrid (BM25 + BioDPR),0.115779,0.437638,0.316,0.111405,0.282061,35.0,15.0,0.042182,17.0,...,0.950367,16.0,11.0,0.357596,17.0,23.0,4.293513e-01,20.0,21.0,0.488975


...



Results for NARRATIVE Queries:


C:\Users\zosia\anaconda3\Lib\site-packages\pyterrier\_evaluation\_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #5: Hybrid (BM25 + BioDPR) (<pyterrier._ops.Sum object at 0x0000022818C1BCE0>)
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.075474,0.398526,0.276,0.086866,0.224339,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.044626,0.220276,0.132,0.067869,0.099810,6.0,44.0,0.000002,12.0,...,0.002098,7.0,22.0,0.000830,8.0,37.0,0.000010,9.0,30.0,0.000069
2,QL (JM),0.058445,0.296964,0.188,0.074529,0.167937,9.0,41.0,0.000184,12.0,...,0.020902,4.0,18.0,0.002918,14.0,32.0,0.030330,12.0,24.0,0.012988
3,BM25 + RM3,0.083607,0.366773,0.216,0.085393,0.201771,21.0,29.0,0.227443,9.0,...,0.121168,4.0,14.0,0.020579,16.0,27.0,0.757486,14.0,23.0,0.142053
4,BioDPR,0.073757,0.402035,0.264,0.091069,0.242682,21.0,29.0,0.827853,21.0,...,0.952202,16.0,15.0,0.745559,22.0,24.0,0.523314,26.0,17.0,0.533645
5,Hybrid (BM25 + BioDPR),0.107555,0.477513,0.316,0.107801,0.283547,43.0,7.0,0.000002,22.0,...,0.035706,14.0,7.0,0.142004,35.0,8.0,0.000279,28.0,12.0,0.013258
